# 02 · Bronze: reclamações como vieram

Lê os 68 arquivos mensais (`.csv.gz`) do volume `mvp_reclamacoes.bronze.arquivos` e grava a tabela Delta `mvp_reclamacoes.bronze.reclamacoes`, **sem alterar o conteúdo**: todas as linhas, todos os setores, todas as colunas publicadas, todas como texto.

**Decisões**
- **Tudo como `STRING`:** a Databricks recomenda *"storing most fields as string… to protect against unexpected schema changes"* ([Medallion](https://docs.databricks.com/aws/en/lakehouse/medallion)).
- **Esquema explícito de 21 colunas, lido por posição:**
  - o mês 2025-09 veio com 2 colunas extras no fim;
  - nos outros 67 meses, o Spark preenche essas 2 com `null` (*"When it meets a record having fewer tokens than the length of the schema, sets null to extra fields"*, [CSV Files](https://spark.apache.org/docs/latest/sql-data-sources-csv.html));
  - ler por posição é seguro porque `scripts/baixar_reclamacoes.py` confere que as 19 primeiras colunas de cada mês estão na ordem padrão.
- **Nomes de coluna originais:** as colunas mantêm os nomes do CSV, com espaço e acento, graças ao column mapping do Delta ([Column mapping](https://docs.databricks.com/aws/en/delta/column-mapping)). A renomeação fica para a Silver.
- **Valores preservados:** espaços não são removidos (`ignoreLeadingWhiteSpace` e `ignoreTrailingWhiteSpace` são `false` na leitura).
  - Única diferença de representação: **um campo vazio no CSV vira `NULL`** (o padrão de `nullValue` é a string vazia).
- **Colunas de controle** (não vieram da fonte, por isso o prefixo `_`):
  - `_arquivo_origem`: nome do arquivo de origem;
  - `_data_ingestao`: momento da carga.
- **Carga:** a tabela é recriada a cada execução (`CREATE OR REPLACE`), porque o dado é um lote histórico fechado.

## 1. Parâmetros e esquema

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

CAMINHO_ARQUIVOS = "/Volumes/mvp_reclamacoes/bronze/arquivos/reclamacoes/"
TABELA = "mvp_reclamacoes.bronze.reclamacoes"
ARQUIVO_COM_EXTRAS = "finalizadas_2025-09.csv.gz"

COLUNAS = [
    "Região", "UF", "Cidade", "Sexo", "Faixa Etária", "Data Finalização", "Tempo Resposta",
    "Nome Fantasia", "Segmento de Mercado", "Área", "Assunto", "Grupo Problema", "Problema",
    "Como Comprou Contratou", "Procurou Empresa", "Respondida", "Situação",
    "Avaliação Reclamação", "Nota do Consumidor",
    # Só existem no arquivo de 2025-09; nos demais meses ficam nulas.
    "Interação com Judiciario", "Último Complemento Consumidor",
]
esquema = StructType([StructField(coluna, StringType()) for coluna in COLUNAS])

## 2. Leitura dos arquivos

In [ ]:
reclamacoes = (
    spark.read
    .schema(esquema)
    .option("header", True)  # pula a primeira linha de cada arquivo
    .option("sep", ";")
    .csv(CAMINHO_ARQUIVOS)
    .select(
        "*",
        F.col("_metadata.file_name").alias("_arquivo_origem"),
        F.current_timestamp().alias("_data_ingestao"),
    )
)

## 3. Gravação da tabela Delta

O column mapping é ativado já na criação, para aceitar os nomes com espaço.

In [ ]:
reclamacoes.createOrReplaceTempView("reclamacoes_lidas")
spark.sql(f"""
    CREATE OR REPLACE TABLE {TABELA}
    TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
    AS SELECT * FROM reclamacoes_lidas
""")

## 4. Verificações

1. **Nenhuma linha perdida ou fundida:** linhas da tabela = linhas de texto dos arquivos − 1 cabeçalho por arquivo. A contagem de texto usa `spark.read.text`, que não interpreta o CSV.
2. **Todos os arquivos entraram:** 68 arquivos distintos em `_arquivo_origem`.
3. **Colunas alinhadas:** as 2 colunas extras vêm preenchidas em todas as linhas de 2025-09 e em nenhuma linha dos outros meses.

In [ ]:
tabela = spark.table(TABELA)
extras = F.col("Interação com Judiciario").isNotNull() | F.col("Último Complemento Consumidor").isNotNull()
de_2025_09 = F.col("_arquivo_origem") == ARQUIVO_COM_EXTRAS

texto = spark.read.text(CAMINHO_ARQUIVOS).select(F.col("_metadata.file_name").alias("arquivo"))
linhas_texto, arquivos_texto = texto.count(), texto.select("arquivo").distinct().count()
linhas_tabela = tabela.count()
arquivos_tabela = tabela.select("_arquivo_origem").distinct().count()
extras_fora_de_2025_09 = tabela.filter(extras & ~de_2025_09).count()
sem_extra_em_2025_09 = tabela.filter(de_2025_09 & F.col("Interação com Judiciario").isNull()).count()

print(f"linhas na tabela: {linhas_tabela:,} | linhas de texto: {linhas_texto:,} | arquivos: {arquivos_texto}")
print(f"arquivos distintos na tabela: {arquivos_tabela}")
print(f"linhas fora de 2025-09 com colunas extras: {extras_fora_de_2025_09}")
print(f"linhas de 2025-09 sem 'Interação com Judiciario': {sem_extra_em_2025_09}")

assert linhas_tabela == linhas_texto - arquivos_texto, "número de linhas diverge do texto dos arquivos"
assert arquivos_tabela == arquivos_texto == 68, "esperados 68 arquivos"
assert extras_fora_de_2025_09 == 0, "colunas extras preenchidas fora de 2025-09: há linhas desalinhadas"
assert sem_extra_em_2025_09 == 0, "linhas de 2025-09 sem as colunas extras: há linhas desalinhadas"
print("Todas as verificações passaram.")

## 5. Registros por arquivo e amostra

In [ ]:
# print em vez de display: mostra os 68 arquivos inteiros, também no kernel local.
contagem = tabela.groupBy("_arquivo_origem").count().orderBy("_arquivo_origem").collect()
for linha in contagem:
    print(f"{linha['_arquivo_origem']}  {linha['count']:>9,}")
print(f"total: {sum(linha['count'] for linha in contagem):,} registros em {len(contagem)} arquivos")

In [ ]:
%sql
SELECT * FROM mvp_reclamacoes.bronze.reclamacoes LIMIT 5